In [1]:
# 01 - Exploratory Data Analysis
## Dataset exploration, sales analysis, promotion analysis, and visualization

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (14, 6)
%matplotlib inline

DATA_DIR = '../data/raw/'

transactions = pd.read_csv(f'{DATA_DIR}transaction_data.csv')
products = pd.read_csv(f'{DATA_DIR}product.csv')
campaign_desc = pd.read_csv(f'{DATA_DIR}campaign_desc.csv')
campaign_table = pd.read_csv(f'{DATA_DIR}campaign_table.csv')
coupons = pd.read_csv(f'{DATA_DIR}coupon.csv')
coupon_redempt = pd.read_csv(f'{DATA_DIR}coupon_redempt.csv')
causal = pd.read_csv(f'{DATA_DIR}causal_data.csv')
hh_demo = pd.read_csv(f'{DATA_DIR}hh_demographic.csv')

print('=== Dataset Shapes ===')
print(f'Transactions: {transactions.shape}')
print(f'Products: {products.shape}')
print(f'Campaign Desc: {campaign_desc.shape}')
print(f'Campaign Table: {campaign_table.shape}')
print(f'Coupons: {coupons.shape}')
print(f'Coupon Redemptions: {coupon_redempt.shape}')
print(f'Causal Data: {causal.shape}')
print(f'Household Demographics: {hh_demo.shape}')

print('=== Transaction Columns ===')
print(transactions.columns.tolist())
print('\n=== Product Columns ===')
print(products.columns.tolist())

transactions.head()

products.head()

campaign_desc.head(10)

## Sales Analysis

daily_sales = transactions.groupby('DAY')['SALES_VALUE'].sum().reset_index()
daily_sales.columns = ['day', 'sales']

plt.figure(figsize=(14, 6))
plt.plot(daily_sales['day'], daily_sales['sales'], alpha=0.7)
plt.title('Daily Sales Over Time')
plt.xlabel('Day')
plt.ylabel('Total Sales ($)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

weekly_sales = transactions.groupby('WEEK_NO')['SALES_VALUE'].sum().reset_index()
weekly_sales.columns = ['week', 'sales']

plt.figure(figsize=(14, 6))
plt.bar(weekly_sales['week'], weekly_sales['sales'], alpha=0.7)
plt.title('Weekly Sales Over Time')
plt.xlabel('Week')
plt.ylabel('Total Sales ($)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

merged = transactions.merge(products[['PRODUCT_ID', 'DEPARTMENT', 'COMMODITY_DESC']], on='PRODUCT_ID', how='left')
dept_sales = merged.groupby('DEPARTMENT')['SALES_VALUE'].sum().sort_values(ascending=False).head(15)

plt.figure(figsize=(12, 6))
dept_sales.plot(kind='bar')
plt.title('Top 15 Departments by Sales')
plt.xlabel('Department')
plt.ylabel('Total Sales ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Promotion Analysis

print('=== Campaigns Overview ===')
campaign_desc['duration'] = campaign_desc['END_DAY'] - campaign_desc['START_DAY'] + 1
print(campaign_desc[['CAMPAIGN', 'DESCRIPTION', 'START_DAY', 'END_DAY', 'duration']].to_string())

discount_data = transactions[transactions['RETAIL_DISC'] < 0].copy()
discount_data['discount_pct'] = abs(discount_data['RETAIL_DISC']) / (discount_data['SALES_VALUE'] + abs(discount_data['RETAIL_DISC']))

print(f'Total transactions with discount: {len(discount_data):,}')
print(f'Discount rate stats:')
print(discount_data['discount_pct'].describe())

plt.figure(figsize=(10, 4))
plt.hist(discount_data['discount_pct'], bins=50, alpha=0.7, edgecolor='black')
plt.title('Distribution of Discount Rates')
plt.xlabel('Discount Percentage')
plt.ylabel('Frequency')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

camp_merged = campaign_desc.merge(campaign_table, on='CAMPAIGN', how='left')
print('Campaign-product mapping:')
print(camp_merged.groupby('CAMPAIGN')['PRODUCT_ID'].nunique().describe())

# Sales around campaign periods
camp = campaign_desc.iloc[0]
start, end = camp['START_DAY'], camp['END_DAY']

window = transactions[(transactions['DAY'] >= start - 14) & (transactions['DAY'] <= end + 14)]
window_sales = window.groupby('DAY')['SALES_VALUE'].sum().reset_index()

plt.figure(figsize=(12, 5))
plt.plot(window_sales['DAY'], window_sales['SALES_VALUE'], marker='o')
plt.axvline(x=start, color='green', linestyle='--', label='Campaign Start')
plt.axvline(x=end, color='red', linestyle='--', label='Campaign End')
plt.title(f'Campaign {camp["CAMPAIGN"]}: Sales Around Promotion Period')
plt.xlabel('Day')
plt.ylabel('Sales ($)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('=== Key Insights ===')
print(f'Date range: {transactions["DAY"].min()} to {transactions["DAY"].max()}')
print(f'Number of products: {products["PRODUCT_ID"].nunique():,}')
print(f'Number of campaigns: {campaign_desc["CAMPAIGN"].nunique()}')
print(f'Total sales: ${transactions["SALES_VALUE"].sum():,.2f}')
print(f'Total transactions: {len(transactions):,}')


KeyboardInterrupt: 